# 🧠 Colab → Hermes: LLM-сервер (vLLM)

Этот ноутбук поднимает на видеокарте Google Colab **настоящий LLM-сервер** (открытая модель Qwen2.5), делает для него публичную ссылку и сам себя проверяет. Ссылку потом подключаем к Hermes — и модель из Colab доступна как обычная модель.

**Как запустить:**
1. Меню **Среда выполнения → Сменить тип среды выполнения → аппаратный ускоритель: T4 GPU** → Сохранить.
2. Меню **Среда выполнения → Выполнить все** (или Ctrl+F9). Появится предупреждение, что ноутбук «не от Google» — жми **Run anyway**.
3. Первый запуск: 5–10 минут (ставятся библиотеки, качается модель ~5 ГБ). Дождись в конце блока **«СКОПИРУЙ ЭТО…»** и пришли его в чат Hermes.

**Что важно знать:**
- Бесплатная Colab иногда не даёт GPU. Если в ячейке «2. Проверка GPU» написано «НЕ НАЙДЕН» — проверь шаг 1.
- Сессия живёт до ~12 часов (обычно меньше). Умерла сессия → просто «Выполнить все» заново, ссылка будет новая.
- Ноутбук — на T4 (16 ГБ) крутится модель 7B. На платных GPU (L4 / A100) ноутбук сам выберет модель крупнее.
- Одновременно только один GPU-ноутбук: либо этот, либо «картиночный». Оба сразу — только на платных GPU.

**Пресеты моделей (меняются в ячейке 1):**

| пресет | модель | для кого |
|---|---|---|
| `auto` | 7B / 14B / 32B по видеопамяти | по умолчанию |
| `7b` | Qwen2.5-7B-Instruct-AWQ | T4 (бесплатная) |
| `14b` | Qwen2.5-14B-Instruct-AWQ | L4 / A100 |
| `32b` | Qwen2.5-32B-Instruct-AWQ | A100 40GB |
| `vision` | Qwen2.5-VL-7B (понимает картинки) | T4 |

Если vLLM категорически не заводится на T4 — в конце есть запасная ячейка **«АЛЬТЕРНАТИВА: Ollama»**, выполни только её.

In [ ]:
# ==== НАСТРОЙКИ (обычно менять не нужно) ====
MODEL_PRESET = "auto"   # "auto" | "7b" | "14b" | "32b" | "vision"
MAX_MODEL_LEN = 32768   # контекст в токенах (снизь до 16384, если совсем мало видеопамяти)
PORT = 8000             # локальный порт сервера

print("Настройки загружены. Пресет модели:", MODEL_PRESET)

In [ ]:
import subprocess, re

gpu_info = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
    shell=True, capture_output=True, text=True,
).stdout.strip()
print("GPU:", gpu_info if gpu_info else "НЕ НАЙДЕН")
if not gpu_info:
    raise RuntimeError(
        "Нет GPU. Сделай: Среда выполнения → Сменить тип среды выполнения → T4 GPU → Сохранить, "
        "затем: Среда выполнения → Выполнить все."
    )

m = re.search(r"([0-9]+)\s*MiB", gpu_info)
VRAM_GB = (int(m.group(1)) / 1024.0) if m else 16.0

PRESETS = {
    "7b":     ("Qwen/Qwen2.5-7B-Instruct-AWQ",    32768),
    "14b":    ("Qwen/Qwen2.5-14B-Instruct-AWQ",   32768),
    "32b":    ("Qwen/Qwen2.5-32B-Instruct-AWQ",   32768),
    "vision": ("Qwen/Qwen2.5-VL-7B-Instruct-AWQ", 32768),
}
if MODEL_PRESET == "auto":
    _preset = "7b" if VRAM_GB < 20 else ("14b" if VRAM_GB < 38 else "32b")
else:
    _preset = MODEL_PRESET

MODEL_ID, _native_len = PRESETS[_preset]
MAX_LEN = min(MAX_MODEL_LEN, _native_len)
MODEL_NAME = "colab-llm"
print("Видеопамять ≈ %.0f ГБ → пресет '%s' → %s (контекст %d)" % (VRAM_GB, _preset, MODEL_ID, MAX_LEN))

In [ ]:
import subprocess, sys

print("Ставлю vLLM (2–5 минут, один раз за сессию). Подожди, ничего не нажимай...")
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "vllm", "huggingface_hub[hf_transfer]"],
    capture_output=True, text=True, timeout=1800,
)
if r.returncode == 0:
    print("✅ vLLM установлен")
else:
    print("❌ pip завершился с ошибкой. Хвост вывода:")
    print((r.stdout or "")[-1500:])
    print((r.stderr or "")[-1500:])
    raise RuntimeError("Не удалось установить vLLM — попробуй ячейку «АЛЬТЕРНАТИВА: Ollama» в конце ноутбука.")

In [ ]:
import os, sys, time, subprocess, secrets

API_KEY = globals().get("API_KEY") or ("sk-colab-" + secrets.token_hex(12))
os.makedirs("/content/logs", exist_ok=True)
VLLM_LOG = "/content/logs/vllm.log"

env = os.environ.copy()
env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
env["VLLM_NO_USAGE_STATS"] = "1"

cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--served-model-name", MODEL_NAME,
    "--dtype", "float16",
    "--max-model-len", str(MAX_LEN),
    "--gpu-memory-utilization", "0.90",
    "--api-key", API_KEY,
    "--enable-auto-tool-choice",
    "--tool-call-parser", "hermes",
    "--enforce-eager",
    "--port", str(PORT),
]
if _preset == "vision":
    cmd += ["--limit-mm-per-prompt", "image=4"]

print("Запускаю сервер. Первый раз качается модель (~5 ГБ) — это 2–6 минут. Жди...")
with open(VLLM_LOG, "w") as lf:
    proc = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, env=env)

deadline = time.time() + 1200
ok = False
while time.time() < deadline:
    log = open(VLLM_LOG, errors="ignore").read()
    if ("Application startup complete" in log) or ("Uvicorn running on" in log):
        ok = True
        break
    if proc.poll() is not None:
        break
    time.sleep(5)

if ok:
    print("✅ LLM-сервер запущен локально на порту", PORT)
else:
    print("❌ Сервер не стартовал. Последние строки лога:")
    print(open(VLLM_LOG, errors="ignore").read()[-2500:])
    raise RuntimeError(
        "vLLM не запустился. Варианты: 1) выполни ячейку «АЛЬТЕРНАТИВА: Ollama» в конце; "
        "2) Среда выполнения → Перезапустить сессию → Выполнить все ещё раз."
    )

In [ ]:
import subprocess, os, re, time, json, urllib.request

# 1) cloudflared — быстрый туннель к локальному порту (скачивается один раз)
if not os.path.exists("/content/cloudflared"):
    subprocess.run(
        "wget -q -O /content/cloudflared "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "&& chmod +x /content/cloudflared",
        shell=True, check=True,
    )

os.makedirs("/content/logs", exist_ok=True)
TUN_LOG = "/content/logs/cloudflared.log"
with open(TUN_LOG, "w") as lf:
    subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:%d" % PORT, "--no-autoupdate"],
        stdout=lf, stderr=subprocess.STDOUT,
    )

url = None
for _ in range(60):
    mm = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TUN_LOG, errors="ignore").read())
    if mm:
        url = mm.group(0)
        break
    time.sleep(2)

if not url:
    print(open(TUN_LOG, errors="ignore").read()[-1500:])
    raise RuntimeError("Туннель не поднялся — запусти эту ячейку ещё раз")

print("Публичный адрес:", url)
time.sleep(3)

# 2) самопроверка: список моделей + реальный чат через туннель
req = urllib.request.Request(url + "/v1/models", headers={"Authorization": "Bearer " + API_KEY})
models = json.load(urllib.request.urlopen(req, timeout=30))
print("Модели на сервере:", [x["id"] for x in models.get("data", [])])

payload = json.dumps({
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Ответь коротко: ты работаешь?"}],
    "max_tokens": 50,
}).encode()
req = urllib.request.Request(
    url + "/v1/chat/completions", data=payload,
    headers={"Authorization": "Bearer " + API_KEY, "Content-Type": "application/json"},
)
resp = json.load(urllib.request.urlopen(req, timeout=300))
print("Ответ модели:", resp["choices"][0]["message"]["content"])

print()
print("=" * 56)
print("СКОПИРУЙ ЭТО И ПРИШЛИ В ЧАТ HERMES")
print("URL: " + url)
print("KEY: " + API_KEY)
print("MODEL: " + MODEL_NAME)
print("=" * 56)

---
## АЛЬТЕРНАТИВА: Ollama (если vLLM не завёлся)

Если предыдущие ячейки упали с ошибкой — выполни **только эту ячейку** (она самодостаточная: поставит Ollama, скачает модель 7B и поднимет туннель). Учти: перед этим лучше сделать **Среда выполнения → Перезапустить сессию**.

In [ ]:
# ==== АЛЬТЕРНАТИВА: Ollama. Выполни ЭТУ ячейку, если vLLM не завёлся. ====
import subprocess, os, sys, time, re, json, secrets, urllib.request

API_KEY = "sk-colab-" + secrets.token_hex(12)
MODEL_NAME = "qwen2.5:7b-instruct-q4_K_M"
os.makedirs("/content/logs", exist_ok=True)

print("Ставлю Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

os.makedirs("/content/ollama_models", exist_ok=True)
env = os.environ.copy()
env["OLLAMA_MODELS"] = "/content/ollama_models"
env["OLLAMA_HOST"] = "127.0.0.1:%d" % PORT
with open("/content/logs/ollama.log", "w") as lf:
    subprocess.Popen(["ollama", "serve"], stdout=lf, stderr=subprocess.STDOUT, env=env)
time.sleep(5)

print("Качаю модель qwen2.5 7B (~4.7 ГБ, 2–4 минуты)...")
subprocess.run(["ollama", "pull", MODEL_NAME], check=True, env=env)

if not os.path.exists("/content/cloudflared"):
    subprocess.run(
        "wget -q -O /content/cloudflared "
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "&& chmod +x /content/cloudflared",
        shell=True, check=True,
    )
with open("/content/logs/cloudflared.log", "w") as lf:
    subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:%d" % PORT, "--no-autoupdate"],
        stdout=lf, stderr=subprocess.STDOUT,
    )

url = None
for _ in range(60):
    mm = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/logs/cloudflared.log", errors="ignore").read())
    if mm:
        url = mm.group(0)
        break
    time.sleep(2)
if not url:
    raise RuntimeError("Туннель не поднялся — запусти эту ячейку ещё раз")

time.sleep(3)
payload = json.dumps({
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Ответь коротко: ты работаешь?"}],
    "max_tokens": 50,
}).encode()
req = urllib.request.Request(url + "/v1/chat/completions", data=payload, headers={"Content-Type": "application/json"})
resp = json.load(urllib.request.urlopen(req, timeout=300))
print("Ответ модели:", resp["choices"][0]["message"]["content"])

print()
print("=" * 56)
print("СКОПИРУЙ ЭТО И ПРИШЛИ В ЧАТ HERMES")
print("URL: " + url)
print("KEY: " + API_KEY)
print("MODEL: " + MODEL_NAME)
print("=" * 56)

---
## Если что-то не так

- **«Нет GPU»** → Среда выполнения → Сменить тип среды выполнения → **T4 GPU** → Сохранить, и «Выполнить все» заново.
- **vLLM упал** → запасная ячейка «АЛЬТЕРНАТИВА: Ollama» (или Перезапустить сессию и ещё раз).
- **Colab пишет, что функция недоступна / не даёт GPU совсем** → включи VPN и перезапусти сессию.
- **Хочу другую модель** → поменяй `MODEL_PRESET` в ячейке 1 и «Выполнить все» (модель скачается заново).
- **Ссылка перестала работать** → сессия Colab умерла. Это норма: «Выполнить все» заново и пришли новый блок в Hermes.